# Log Streaming & Artifact Retrieval

Real-time visibility separates a platform from a black box. A job that disappears into a remote machine and returns silence is not useful. In this notebook, log lines produced on a remote machine stream back in real time, are fanned out to all connected dashboard clients over WebSocket, and persisted for later retrieval. When the job finishes, artifacts are pulled back automatically.

The components built here are: a `LogEvent` schema, a subprocess capture harness, a WebSocket fan-out manager, a persistence layer backed by newline-delimited JSON, an artifact retrieval pipeline, a dev-mode file watcher, and a Flet log viewer that connects everything into the dashboard.

## Structured Log Events

Raw strings are hard to filter, route, and persist. We model every log line as a first-class `LogEvent` object with a severity level, originating job and machine, a human-readable message, and a timestamp. This makes it trivial to colorize output by level, filter noise in the viewer, and query history by severity.

The **log level** hierarchy is:

In [ ]:
from enum import Enum

class LogLevel(str, Enum):
    DEBUG   = "debug"
    INFO    = "info"
    WARNING = "warning"
    ERROR   = "error"

Inheriting from `str` means `LogLevel` members serialize to plain strings, which is convenient for JSON and for comparisons in conditional logic without calling `.value`.

The `LogEvent` model:

In [ ]:
from datetime import datetime, timezone
from pydantic import BaseModel, Field

class LogEvent(BaseModel):
    job_id:     str
    machine_id: str
    level:      LogLevel
    message:    str
    timestamp:  datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

Serialization round-trip to verify the schema:

In [ ]:
import json

event = LogEvent(
    job_id="job-001",
    machine_id="gpu-a100-01",
    level=LogLevel.INFO,
    message="Epoch 1/10 — loss: 0.4231",
)

serialized   = event.model_dump_json()
deserialized = LogEvent.model_validate_json(serialized)

print(serialized)
print(deserialized.level, type(deserialized.level))
print("round-trip OK:", event == deserialized)

:::{.callout-note}
We store the timestamp in UTC and use `datetime` (not `str`) so that readers can sort events across machines whose clocks may drift slightly. The Flet viewer renders it in local time.

:::

## Subprocess stdout/stderr Capture

When a job runs as a subprocess on the platform machine, we want to capture both `stdout` and `stderr` line by line as they are produced, tag each line with the appropriate `LogLevel`, and emit `LogEvent` objects in real time rather than waiting for the process to finish.

`asyncio.create_subprocess_exec` with `PIPE` for both streams, combined with `asyncio.gather` on two concurrent readers, gives us interleaved real-time capture without blocking:

$$\text{stdout} \xrightarrow{\text{reader task}} \text{INFO events}$$
$$\text{stderr} \xrightarrow{\text{reader task}} \text{ERROR events}$$

In [ ]:
import asyncio
from asyncio import StreamReader
from typing import AsyncIterator

async def _read_stream(
    stream: StreamReader,
    level: LogLevel,
    job_id: str,
    machine_id: str,
    events: list[LogEvent],
) -> None:
    """Read lines from a subprocess stream and append LogEvents."""
    async for raw in stream:
        line = raw.decode().rstrip("\n")
        if line:
            events.append(LogEvent(
                job_id=job_id,
                machine_id=machine_id,
                level=level,
                message=line,
            ))


async def capture_subprocess(
    cmd: list[str],
    job_id: str,
    machine_id: str,
) -> list[LogEvent]:
    """Run cmd and collect all stdout/stderr lines as LogEvents."""
    proc = await asyncio.create_subprocess_exec(
        *cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )

    events: list[LogEvent] = []
    await asyncio.gather(
        _read_stream(proc.stdout, LogLevel.INFO,  job_id, machine_id, events),
        _read_stream(proc.stderr, LogLevel.ERROR, job_id, machine_id, events),
    )

    await proc.wait()
    return events

Running against a small shell script that writes to both streams:

In [ ]:
events = await capture_subprocess(
    ["bash", "-c", "echo 'starting'; echo 'warn' >&2; echo 'done'"],
    job_id="job-demo",
    machine_id="local",
)

for e in events:
    print(f"[{e.level.upper():7}] {e.message}")

:::{.callout-caution}
The ordering of `stdout` and `stderr` events is non-deterministic when both streams produce output simultaneously. This is inherent to `asyncio.gather` — both readers race. If strict ordering is required, merge to a single stream at the shell level with `2>&1`.

:::

In production the runner would emit events incrementally via a callback or an async generator rather than collecting them into a list. We use a list here for testability.

## WebSocket Log Fan-out

A single job may be watched by multiple clients simultaneously — the submitting user, a team member, and the dashboard all connect to the same stream. The fan-out pattern routes each `LogEvent` to every active WebSocket connection subscribed to that job.

We implement this with a `ConnectionManager` that maps `job_id → set[WebSocket]`:

In [ ]:
from fastapi import WebSocket

class ConnectionManager:
    """Fan-out manager: route LogEvents to all WebSocket subscribers of a job."""

    def __init__(self) -> None:
        self._subs: dict[str, set[WebSocket]] = {}

    async def connect(self, job_id: str, ws: WebSocket) -> None:
        await ws.accept()
        self._subs.setdefault(job_id, set()).add(ws)

    def disconnect(self, job_id: str, ws: WebSocket) -> None:
        subs = self._subs.get(job_id, set())
        subs.discard(ws)
        if not subs:
            self._subs.pop(job_id, None)

    async def broadcast(self, event: LogEvent) -> None:
        """Send event JSON to every subscriber of event.job_id."""
        payload = event.model_dump_json()
        dead: list[WebSocket] = []

        for ws in list(self._subs.get(event.job_id, set())):
            try:
                await ws.send_text(payload)
            except Exception:                       # <1>
                dead.append(ws)

        for ws in dead:                             # <2>
            self.disconnect(event.job_id, ws)


manager = ConnectionManager()

1. A send failure means the client has disconnected without a clean close frame — we collect it and prune rather than raising immediately.
2. Dead connections are removed after iteration so we never modify the set while iterating it.

The FastAPI WebSocket endpoint wires `ConnectionManager` into the API:

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.websocket("/jobs/{job_id}/logs/ws")
async def ws_logs(job_id: str, websocket: WebSocket):
    await manager.connect(job_id, websocket)
    try:
        while True:                                 # <1>
            await websocket.receive_text()
    except Exception:
        pass
    finally:
        manager.disconnect(job_id, websocket)       # <2>

1. We keep the connection open by awaiting client messages (pings or control frames). The server never sends from here — it sends via `manager.broadcast` called from the job runner.
2. The `finally` block guarantees cleanup even if the client disconnects abruptly.

To exercise the fan-out without a live runner, we write a mock emitter. The server runs in a background thread and an `httpx` async WebSocket client connects to it:

In [ ]:
import threading
import uvicorn
import httpx

# Background server
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8700, log_level="error")

t = threading.Thread(target=run_server, daemon=True)
t.start()

await asyncio.sleep(0.5)  # let the server start

# Mock emitter — broadcasts 5 events directly via manager
async def mock_emit(job_id: str) -> None:
    for i in range(5):
        await asyncio.sleep(0.05)
        await manager.broadcast(LogEvent(
            job_id=job_id,
            machine_id="local",
            level=LogLevel.INFO,
            message=f"Step {i+1}/5 complete",
        ))

# Client receives streamed events
received: list[str] = []

async def ws_client(job_id: str) -> None:
    async with httpx.AsyncClient() as client:
        async with client.stream("GET", f"ws://127.0.0.1:8700/jobs/{job_id}/logs/ws",
                                  headers={"upgrade": "websocket"}):
            pass  # httpx WS not supported; see note below

# Use websockets library instead
import websockets

async def ws_client_ws(job_id: str) -> None:
    uri = f"ws://127.0.0.1:8700/jobs/{job_id}/logs/ws"
    async with websockets.connect(uri) as ws:
        async for msg in ws:
            event = LogEvent.model_validate_json(msg)
            received.append(event.message)
            if len(received) >= 5:
                break

await asyncio.gather(
    mock_emit("job-ws-demo"),
    ws_client_ws("job-ws-demo"),
)

print("\n".join(received))

**NOTE:** `httpx` does not natively support the WebSocket protocol — we use the `websockets` library for the client here. In production the dashboard connects via the browser's native `WebSocket` API, and CLI tools use `websockets` or `websocket-client`.

## Log Persistence

Streaming alone is ephemeral — once a subscriber disconnects, the history is gone. We persist every `LogEvent` to a newline-delimited JSON file at `logs/{job_id}.jsonl`. The format is append-friendly: each write is a single `model_dump_json()` call followed by a newline, and reading is a line-by-line scan with no parsing of a full JSON array.

The persistence helper:

In [ ]:
import aiofiles
from pathlib import Path

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

async def persist_event(event: LogEvent) -> None:
    path = LOG_DIR / f"{event.job_id}.jsonl"
    async with aiofiles.open(path, "a") as f:
        await f.write(event.model_dump_json() + "\n")


async def read_events(job_id: str) -> list[LogEvent]:
    path = LOG_DIR / f"{job_id}.jsonl"
    if not path.exists():
        return []
    events = []
    async with aiofiles.open(path, "r") as f:
        async for line in f:
            line = line.strip()
            if line:
                events.append(LogEvent.model_validate_json(line))
    return events

Write/read round-trip:

In [ ]:
import shutil

test_events = [
    LogEvent(job_id="job-persist", machine_id="local", level=LogLevel.INFO,    message="start"),
    LogEvent(job_id="job-persist", machine_id="local", level=LogLevel.WARNING, message="slow epoch"),
    LogEvent(job_id="job-persist", machine_id="local", level=LogLevel.INFO,    message="done"),
]

for e in test_events:
    await persist_event(e)

recovered = await read_events("job-persist")
print(f"wrote {len(test_events)}, read back {len(recovered)}")
for e in recovered:
    print(f"  [{e.level}] {e.message}")

# cleanup
shutil.rmtree(LOG_DIR)

The HTTP endpoint that serves stored logs for a completed job:

In [ ]:
from fastapi.responses import JSONResponse

@app.get("/jobs/{job_id}/logs")
async def get_logs(job_id: str):
    events = await read_events(job_id)
    return JSONResponse([json.loads(e.model_dump_json()) for e in events])

:::{.callout-note}
In production `persist_event` is called inside `manager.broadcast` so that every fanned-out event is also written to disk atomically from the same coroutine. This ensures the persisted log and the live stream stay in sync.

:::

## Artifact Retrieval

When a job finishes, it typically produces **artifacts** — trained model checkpoints, evaluation metrics, generated files. The platform must pull these back from the remote machine and make them available via the API. We define an `ArtifactManifest` that lists what was produced and where it was saved:

In [ ]:
from pydantic import BaseModel

class ArtifactEntry(BaseModel):
    filename:    str
    size_bytes:  int
    remote_path: str
    local_path:  str

class ArtifactManifest(BaseModel):
    job_id:    str
    artifacts: list[ArtifactEntry] = []

The retrieval function copies each artifact from its `remote_path` to `local_path`, emitting a `LogEvent` per file so progress is visible in the live stream. In production `remote_path` is an SSH or S3 path and the copy uses `paramiko` or `boto3`; here we use `shutil.copy` against local paths to keep the demo runnable:

In [ ]:
import os

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

async def fetch_artifacts(
    manifest: ArtifactManifest,
    machine_id: str,
    emit: bool = True,
) -> ArtifactManifest:
    """Copy artifacts to local_path and optionally emit progress events."""
    for entry in manifest.artifacts:
        Path(entry.local_path).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(entry.remote_path, entry.local_path)    # <1>

        if emit:
            event = LogEvent(
                job_id=manifest.job_id,
                machine_id=machine_id,
                level=LogLevel.INFO,
                message=f"artifact retrieved: {entry.filename} ({entry.size_bytes} bytes)",
            )
            await manager.broadcast(event)

    return manifest

1. Replace with `asyncssh.get` or `boto3.download_file` for remote machines.

Runnable demo with temporary files:

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    # Create fake remote artifacts
    remote = Path(tmpdir) / "remote"
    remote.mkdir()
    (remote / "model.pt").write_bytes(b"model" * 20)
    (remote / "metrics.json").write_text('{"acc": 0.97}')

    local_base = Path(tmpdir) / "local"

    manifest = ArtifactManifest(
        job_id="job-art",
        artifacts=[
            ArtifactEntry(
                filename="model.pt",
                size_bytes=(remote / "model.pt").stat().st_size,
                remote_path=str(remote / "model.pt"),
                local_path=str(local_base / "model.pt"),
            ),
            ArtifactEntry(
                filename="metrics.json",
                size_bytes=(remote / "metrics.json").stat().st_size,
                remote_path=str(remote / "metrics.json"),
                local_path=str(local_base / "metrics.json"),
            ),
        ]
    )

    result = await fetch_artifacts(manifest, machine_id="local", emit=False)

    for entry in result.artifacts:
        exists = Path(entry.local_path).exists()
        print(f"{entry.filename:15}  local={exists}  size={entry.size_bytes}B")

The API endpoints that expose artifacts to clients:

In [ ]:
from fastapi.responses import FileResponse

# In-memory store mapping job_id -> ArtifactManifest
_manifests: dict[str, ArtifactManifest] = {}

@app.get("/jobs/{job_id}/artifacts")
async def list_artifacts(job_id: str):
    manifest = _manifests.get(job_id)
    if manifest is None:
        return JSONResponse({"error": "not found"}, status_code=404)
    return JSONResponse(manifest.model_dump())


@app.get("/jobs/{job_id}/artifacts/{filename}")
async def download_artifact(job_id: str, filename: str):
    manifest = _manifests.get(job_id)
    if manifest is None:
        return JSONResponse({"error": "job not found"}, status_code=404)

    for entry in manifest.artifacts:
        if entry.filename == filename:
            path = Path(entry.local_path)
            if not path.exists():
                return JSONResponse({"error": "file missing"}, status_code=404)
            return FileResponse(path, filename=filename)    # <1>

    return JSONResponse({"error": "artifact not found"}, status_code=404)

1. `FileResponse` sets `Content-Disposition: attachment` and streams the file efficiently without loading it into memory.

## Dev-Mode File Watching

During local development, jobs write artifacts to a watched directory rather than a remote machine. `watchfiles.awatch` provides an async iterator that yields change events — `(ChangeType, path)` tuples — whenever files are created, modified, or deleted. We can hook this into the same `LogEvent` pipeline:

$$\text{file change} \xrightarrow{\texttt{awatch}} \text{LogEvent} \xrightarrow{\texttt{broadcast}} \text{subscribers}$$

In [ ]:
import watchfiles

async def watch_artifacts_dir(
    directory: Path,
    job_id: str,
    machine_id: str,
    stop_event: asyncio.Event,
) -> None:
    """Watch a directory and emit LogEvents for each file change."""
    async for changes in watchfiles.awatch(directory, stop_event=stop_event):
        for change, path in changes:
            await manager.broadcast(LogEvent(
                job_id=job_id,
                machine_id=machine_id,
                level=LogLevel.INFO,
                message=f"[watch] {change.name}: {Path(path).name}",
            ))

Triggering the watcher with a temporary directory:

In [ ]:
import tempfile

watch_events: list[str] = []

async def capture_watch_broadcast(event: LogEvent) -> None:
    watch_events.append(event.message)

# Patch broadcast to capture without a live WebSocket
_orig_broadcast = manager.broadcast
manager.broadcast = capture_watch_broadcast

stop = asyncio.Event()

with tempfile.TemporaryDirectory() as tmpdir:
    watch_dir = Path(tmpdir)

    async def write_files():
        await asyncio.sleep(0.1)
        (watch_dir / "checkpoint.pt").write_bytes(b"ckpt")
        await asyncio.sleep(0.1)
        (watch_dir / "metrics.json").write_text('{}')
        await asyncio.sleep(0.1)
        stop.set()

    await asyncio.gather(
        watch_artifacts_dir(watch_dir, "job-watch", "local", stop),
        write_files(),
    )

manager.broadcast = _orig_broadcast
print("\n".join(watch_events))

:::{.callout-note}
`watchfiles` uses OS-native file system events (inotify on Linux, FSEvents on macOS, ReadDirectoryChangesW on Windows). It is far more efficient than polling with `os.stat` in a loop and wakes up only when a change actually occurs.

:::

## Flet Log Viewer

**Task.** Build a `LogViewer` Flet component that connects to the `/jobs/{job_id}/logs/ws` WebSocket endpoint and renders incoming `LogEvent` objects in a scrolling `ListView`, color-coded by level.

Flet UI code runs as a standalone script, not inside a Jupyter notebook. The following is run as `dashboard/log_viewer.py`:

```{.python filename="dashboard/log_viewer.py"}
import queue
import threading
import flet as ft
import websockets
import asyncio
from log_event import LogEvent, LogLevel

LEVEL_COLOR = {
    LogLevel.DEBUG:   ft.Colors.GREY_500,
    LogLevel.INFO:    ft.Colors.WHITE,
    LogLevel.WARNING: ft.Colors.YELLOW_400,
    LogLevel.ERROR:   ft.Colors.RED_400,
}


class LogViewer(ft.Column):
    """Scrolling log panel that streams events from a WebSocket."""

    def __init__(self, job_id: str, ws_base: str = "ws://127.0.0.1:8700"):
        super().__init__(expand=True)
        self.job_id  = job_id
        self.ws_url  = f"{ws_base}/jobs/{job_id}/logs/ws"
        self._q: queue.Queue[LogEvent] = queue.Queue()
        self._list   = ft.ListView(expand=True, auto_scroll=True, spacing=2)
        self.controls = [self._list]

    def did_mount(self):
        threading.Thread(target=self._ws_thread, daemon=True).start()
        self.page.run_task(self._poll_queue)

    def _ws_thread(self):
        """Background thread: receive WebSocket messages and push to queue."""
        async def _recv():
            async with websockets.connect(self.ws_url) as ws:
                async for msg in ws:
                    self._q.put(LogEvent.model_validate_json(msg))

        asyncio.run(_recv())

    async def _poll_queue(self):
        """Drain the queue on the Flet event loop and update the ListView."""
        while True:
            while not self._q.empty():
                event = self._q.get_nowait()
                self._list.controls.append(
                    ft.Text(
                        f"[{event.timestamp:%H:%M:%S}] [{event.level.upper():7}] {event.message}",
                        color=LEVEL_COLOR[event.level],
                        font_family="monospace",
                        size=12,
                    )
                )
            self.page.update()
            await asyncio.sleep(0.1)


def main(page: ft.Page):
    page.title = "Log Viewer"
    page.bgcolor = ft.Colors.GREY_900
    page.padding = 12

    job_id = page.route.lstrip("/") or "job-001"
    page.add(LogViewer(job_id=job_id))


ft.app(target=main)
```

The `queue.Queue` bridge is the key pattern: the `websockets` async loop runs in a dedicated background thread, while the Flet UI runs on its own event loop. The queue is the thread-safe handoff point between them. `_poll_queue` drains it every 100 ms and updates the page.

To integrate into the main dashboard, import `LogViewer` from this module and add it as a tab or panel in `dashboard/main.py`:

```python
from log_viewer import LogViewer

tabs.tabs.append(
    ft.Tab(text="Logs", content=LogViewer(job_id=selected_job_id))
)
```

## Appendix: Job History & Replay {#sec-replay}

Live streaming is useful during execution. But a user who connects *after* a job has finished still wants to see what happened. We add a `JobRecord` that persists job metadata alongside the log file, and a `/replay` endpoint that streams the persisted log with an artificial delay — simulating the original pace of execution.

In [ ]:
from datetime import datetime, timezone
from pydantic import BaseModel
from pathlib import Path
import json

class JobRecord(BaseModel):
    job_id:     str
    machine_id: str
    status:     str          # queued | running | done | failed
    submitted:  datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    started:    datetime | None = None
    finished:   datetime | None = None

JOB_DIR = Path("jobs")
JOB_DIR.mkdir(exist_ok=True)

async def save_job_record(record: JobRecord) -> None:
    path = JOB_DIR / f"{record.job_id}.json"
    async with aiofiles.open(path, "w") as f:
        await f.write(record.model_dump_json(indent=2))


async def load_job_record(job_id: str) -> JobRecord | None:
    path = JOB_DIR / f"{job_id}.json"
    if not path.exists():
        return None
    async with aiofiles.open(path) as f:
        return JobRecord.model_validate_json(await f.read())

Persist and reload a job record:

In [ ]:
rec = JobRecord(job_id="job-hist", machine_id="gpu-v100-02", status="done")
await save_job_record(rec)

loaded = await load_job_record("job-hist")
print(loaded)

shutil.rmtree(JOB_DIR)

The replay endpoint reads the persisted `.jsonl` log and streams each event to the client with a configurable delay between lines, recreating the feel of live execution:

In [ ]:
from fastapi.responses import StreamingResponse

@app.get("/jobs/{job_id}/replay")
async def replay_logs(job_id: str, delay: float = 0.05):
    """Stream persisted logs with artificial inter-line delay."""

    async def _stream():
        events = await read_events(job_id)
        for event in events:
            yield event.model_dump_json() + "\n"
            await asyncio.sleep(delay)              # <1>

    return StreamingResponse(_stream(), media_type="application/x-ndjson")

1. The delay is configurable via query parameter so that automated clients can set `delay=0` for fast replay, while human viewers use the default.

We verify the endpoint by writing a few events, then consuming the streaming response with `httpx`:

In [ ]:
LOG_DIR.mkdir(exist_ok=True)

replay_events = [
    LogEvent(job_id="job-replay", machine_id="local", level=LogLevel.INFO,  message="start"),
    LogEvent(job_id="job-replay", machine_id="local", level=LogLevel.INFO,  message="epoch 1"),
    LogEvent(job_id="job-replay", machine_id="local", level=LogLevel.ERROR, message="nan loss"),
]
for e in replay_events:
    await persist_event(e)

received_replay: list[str] = []
async with httpx.AsyncClient(app=app, base_url="http://test") as client:
    async with client.stream("GET", "/jobs/job-replay/replay?delay=0") as resp:
        async for line in resp.aiter_lines():
            if line:
                e = LogEvent.model_validate_json(line)
                received_replay.append(e.message)

print(received_replay)
shutil.rmtree(LOG_DIR)

**Remark.** `StreamingResponse` with `application/x-ndjson` is the idiomatic FastAPI pattern for server-sent data streams that are not WebSocket-based. Clients read line-by-line using `aiter_lines()` in `httpx` or a `ReadableStream` in the browser's Fetch API. This is simpler than WebSocket when the client only needs to receive, never send.

---

■